In [1]:
# ============================================================
# IMPROVED LATE FUSION
# Optimized fusion weight alpha + optimized decision threshold
# Outer evaluation: LOSO
# Inner optimization: GroupKFold using TRAIN participants only
# Final metric: GLOBAL OOF Balanced Accuracy
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Canonical XGBoost model
# ------------------------------------------------------------

base_xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)


# ------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values


# ------------------------------------------------------------
# 3. Search space
# ------------------------------------------------------------

# alpha = geometric contribution
# 1-alpha = valence contribution

alpha_grid = np.arange(0.00, 1.01, 0.05)

# threshold for final fused probability
threshold_grid = np.arange(0.30, 0.71, 0.02)


# ------------------------------------------------------------
# 4. Outer LOSO
# ------------------------------------------------------------

logo = LeaveOneGroupOut()

all_true = []
all_pred = []
all_prob = []
all_participants = []

selected_alphas = []
selected_thresholds = []

n_folds = logo.get_n_splits(X_geo, y, groups)

print("LOSO folds:", n_folds)


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_geo, y, groups), start=1
):

    # ========================================================
    # OUTER TRAIN / TEST
    # ========================================================

    Xg_train = X_geo.iloc[train_idx]
    Xg_test = X_geo.iloc[test_idx]

    Xv_train = X_val.iloc[train_idx]
    Xv_test = X_val.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]
    groups_test = groups[test_idx]


    # ========================================================
    # INNER GROUPED CV
    #
    # We generate OOF probabilities ONLY from outer-training
    # participants.
    # ========================================================

    unique_train_groups = np.unique(groups_train)

    n_inner_splits = min(5, len(unique_train_groups))

    inner_cv = GroupKFold(n_splits=n_inner_splits)

    inner_geo_prob = np.zeros(len(train_idx))
    inner_val_prob = np.zeros(len(train_idx))


    for inner_train_idx, inner_val_idx in inner_cv.split(
        Xg_train,
        y_train,
        groups_train
    ):

        # ----------------------------------------------------
        # Inner data
        # ----------------------------------------------------

        Xg_inner_train = Xg_train.iloc[inner_train_idx]
        Xg_inner_val = Xg_train.iloc[inner_val_idx]

        Xv_inner_train = Xv_train.iloc[inner_train_idx]
        Xv_inner_val = Xv_train.iloc[inner_val_idx]

        y_inner_train = y_train[inner_train_idx]


        # ----------------------------------------------------
        # Imputation fitted ONLY on inner-training subjects
        # ----------------------------------------------------

        geo_imputer_inner = SimpleImputer(strategy="mean")
        val_imputer_inner = SimpleImputer(strategy="mean")

        Xg_inner_train_imp = geo_imputer_inner.fit_transform(
            Xg_inner_train
        )

        Xg_inner_val_imp = geo_imputer_inner.transform(
            Xg_inner_val
        )

        Xv_inner_train_imp = val_imputer_inner.fit_transform(
            Xv_inner_train
        )

        Xv_inner_val_imp = val_imputer_inner.transform(
            Xv_inner_val
        )


        # ----------------------------------------------------
        # Train the two independent classifiers
        # ----------------------------------------------------

        geo_model_inner = clone(base_xgb_model)
        val_model_inner = clone(base_xgb_model)

        geo_model_inner.fit(
            Xg_inner_train_imp,
            y_inner_train
        )

        val_model_inner.fit(
            Xv_inner_train_imp,
            y_inner_train
        )


        # ----------------------------------------------------
        # Inner validation probabilities
        # ----------------------------------------------------

        inner_geo_prob[inner_val_idx] = (
            geo_model_inner.predict_proba(
                Xg_inner_val_imp
            )[:, 1]
        )

        inner_val_prob[inner_val_idx] = (
            val_model_inner.predict_proba(
                Xv_inner_val_imp
            )[:, 1]
        )


    # ========================================================
    # 5. Optimize alpha + threshold using INNER OOF only
    # ========================================================

    best_score = -np.inf
    best_alpha = None
    best_threshold = None


    for alpha in alpha_grid:

        fused_inner_prob = (
            alpha * inner_geo_prob
            +
            (1 - alpha) * inner_val_prob
        )

        for threshold in threshold_grid:

            fused_inner_pred = (
                fused_inner_prob >= threshold
            ).astype(int)

            score = balanced_accuracy_score(
                y_train,
                fused_inner_pred
            )

            if score > best_score:

                best_score = score
                best_alpha = alpha
                best_threshold = threshold


    selected_alphas.append(best_alpha)
    selected_thresholds.append(best_threshold)


    # ========================================================
    # 6. Refit both stream models using ALL outer training data
    # ========================================================

    geo_imputer = SimpleImputer(strategy="mean")
    val_imputer = SimpleImputer(strategy="mean")

    Xg_train_imp = geo_imputer.fit_transform(Xg_train)
    Xg_test_imp = geo_imputer.transform(Xg_test)

    Xv_train_imp = val_imputer.fit_transform(Xv_train)
    Xv_test_imp = val_imputer.transform(Xv_test)


    geo_model = clone(base_xgb_model)
    val_model = clone(base_xgb_model)

    geo_model.fit(
        Xg_train_imp,
        y_train
    )

    val_model.fit(
        Xv_train_imp,
        y_train
    )


    # ========================================================
    # 7. Predict held-out participant
    # ========================================================

    geo_prob = geo_model.predict_proba(
        Xg_test_imp
    )[:, 1]

    val_prob = val_model.predict_proba(
        Xv_test_imp
    )[:, 1]


    fused_prob = (
        best_alpha * geo_prob
        +
        (1 - best_alpha) * val_prob
    )


    fused_pred = (
        fused_prob >= best_threshold
    ).astype(int)


    # ========================================================
    # Store GLOBAL OOF predictions
    # ========================================================

    all_true.extend(y_test)
    all_pred.extend(fused_pred)
    all_prob.extend(fused_prob)
    all_participants.extend(groups_test)


    if fold % 10 == 0 or fold == n_folds:

        print(
            f"Completed {fold}/{n_folds} | "
            f"alpha={best_alpha:.2f} | "
            f"threshold={best_threshold:.2f} | "
            f"inner BA={best_score:.4f}"
        )


# ============================================================
# 8. GLOBAL OOF evaluation
# ============================================================

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_prob = np.array(all_prob)


accuracy = accuracy_score(
    all_true,
    all_pred
)

balanced_acc = balanced_accuracy_score(
    all_true,
    all_pred
)

precision = precision_score(
    all_true,
    all_pred,
    zero_division=0
)

recall = recall_score(
    all_true,
    all_pred,
    zero_division=0
)

f1 = f1_score(
    all_true,
    all_pred,
    zero_division=0
)


results = pd.DataFrame({
    "Configuration": [
        "Optimized Weighted Late Fusion"
    ],
    "Accuracy": [
        f"{accuracy * 100:.2f}%"
    ],
    "Balanced Accuracy": [
        f"{balanced_acc * 100:.2f}%"
    ],
    "Precision": [
        f"{precision * 100:.2f}%"
    ],
    "Recall": [
        f"{recall * 100:.2f}%"
    ],
    "F1": [
        f"{f1 * 100:.2f}%"
    ],
    "N": [
        len(all_true)
    ]
})

display(results)


# ============================================================
# 9. Examine selected fusion parameters
# ============================================================

print("\nSelected alpha statistics")
print("-------------------------")
print(
    "Mean geometric weight:",
    np.mean(selected_alphas)
)

print(
    "Median geometric weight:",
    np.median(selected_alphas)
)

print(
    "Min geometric weight:",
    np.min(selected_alphas)
)

print(
    "Max geometric weight:",
    np.max(selected_alphas)
)


print("\nSelected threshold statistics")
print("-----------------------------")
print(
    "Mean threshold:",
    np.mean(selected_thresholds)
)

print(
    "Median threshold:",
    np.median(selected_thresholds)
)


# ------------------------------------------------------------
# optional detailed fold table
# ------------------------------------------------------------

parameter_df = pd.DataFrame({
    "Fold": np.arange(
        1,
        len(selected_alphas) + 1
    ),
    "Geometric Weight": selected_alphas,
    "Valence Weight": [
        1 - a for a in selected_alphas
    ],
    "Threshold": selected_thresholds
})

display(parameter_df)

NameError: name 'df' is not defined

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD MASTER FEATURE DATASET
# ============================================================

csv_path = r"C:\Users\hp\StressDetection\master_features_restored.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# DEFINE FEATURE GROUPS
# ============================================================

geo_features = [
    "EAR_mean", "EAR_std", "EAR_min", "EAR_max", "EAR_range",
    "blink_mean", "blink_std", "blink_min", "blink_max", "blink_range",
    "gaze_mean", "gaze_std", "gaze_min", "gaze_max", "gaze_range",
    "brow_mean", "brow_std", "brow_min", "brow_max", "brow_range",
    "lip_mean", "lip_std", "lip_min", "lip_max", "lip_range"
]

valence_features = [
    "valence_mean",
    "valence_std",
    "valence_min",
    "valence_max"
]


# ============================================================
# QUICK CHECK
# ============================================================

required_columns = (
    geo_features
    + valence_features
    + ["participant", "label"]
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("❌ Missing columns:", missing_columns)
else:
    print("✅ All required columns are available")
    print("Geometric features:", len(geo_features))
    print("Valence features:", len(valence_features))
    print("Total fused features:", len(geo_features) + len(valence_features))
    print("Participants:", df["participant"].nunique())
    print("Samples:", len(df))
    print("\nLabel distribution:")
    print(df["label"].value_counts())

Dataset loaded successfully
Shape: (29736, 33)
Columns: ['participant', 'task', 'window_id', 'label', 'EAR_mean', 'EAR_std', 'EAR_min', 'EAR_max', 'EAR_range', 'blink_mean', 'blink_std', 'blink_min', 'blink_max', 'blink_range', 'gaze_mean', 'gaze_std', 'gaze_min', 'gaze_max', 'gaze_range', 'brow_mean', 'brow_std', 'brow_min', 'brow_max', 'brow_range', 'lip_mean', 'lip_std', 'lip_min', 'lip_max', 'lip_range', 'valence_mean', 'valence_std', 'valence_min', 'valence_max']
✅ All required columns are available
Geometric features: 25
Valence features: 4
Total fused features: 29
Participants: 56
Samples: 29736

Label distribution:
label
1    19824
0     9912
Name: count, dtype: int64


In [3]:
# ============================================================
# IMPROVED LATE FUSION
# Optimized fusion weight alpha + optimized decision threshold
# Outer evaluation: LOSO
# Inner optimization: GroupKFold using TRAIN participants only
# Final metric: GLOBAL OOF Balanced Accuracy
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Canonical XGBoost model
# ------------------------------------------------------------

base_xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)


# ------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values


# ------------------------------------------------------------
# 3. Search space
# ------------------------------------------------------------

# alpha = geometric contribution
# 1-alpha = valence contribution

alpha_grid = np.arange(0.00, 1.01, 0.05)

# threshold for final fused probability
threshold_grid = np.arange(0.30, 0.71, 0.02)


# ------------------------------------------------------------
# 4. Outer LOSO
# ------------------------------------------------------------

logo = LeaveOneGroupOut()

all_true = []
all_pred = []
all_prob = []
all_participants = []

selected_alphas = []
selected_thresholds = []

n_folds = logo.get_n_splits(X_geo, y, groups)

print("LOSO folds:", n_folds)


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_geo, y, groups), start=1
):

    # ========================================================
    # OUTER TRAIN / TEST
    # ========================================================

    Xg_train = X_geo.iloc[train_idx]
    Xg_test = X_geo.iloc[test_idx]

    Xv_train = X_val.iloc[train_idx]
    Xv_test = X_val.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]
    groups_test = groups[test_idx]


    # ========================================================
    # INNER GROUPED CV
    #
    # We generate OOF probabilities ONLY from outer-training
    # participants.
    # ========================================================

    unique_train_groups = np.unique(groups_train)

    n_inner_splits = min(5, len(unique_train_groups))

    inner_cv = GroupKFold(n_splits=n_inner_splits)

    inner_geo_prob = np.zeros(len(train_idx))
    inner_val_prob = np.zeros(len(train_idx))


    for inner_train_idx, inner_val_idx in inner_cv.split(
        Xg_train,
        y_train,
        groups_train
    ):

        # ----------------------------------------------------
        # Inner data
        # ----------------------------------------------------

        Xg_inner_train = Xg_train.iloc[inner_train_idx]
        Xg_inner_val = Xg_train.iloc[inner_val_idx]

        Xv_inner_train = Xv_train.iloc[inner_train_idx]
        Xv_inner_val = Xv_train.iloc[inner_val_idx]

        y_inner_train = y_train[inner_train_idx]


        # ----------------------------------------------------
        # Imputation fitted ONLY on inner-training subjects
        # ----------------------------------------------------

        geo_imputer_inner = SimpleImputer(strategy="mean")
        val_imputer_inner = SimpleImputer(strategy="mean")

        Xg_inner_train_imp = geo_imputer_inner.fit_transform(
            Xg_inner_train
        )

        Xg_inner_val_imp = geo_imputer_inner.transform(
            Xg_inner_val
        )

        Xv_inner_train_imp = val_imputer_inner.fit_transform(
            Xv_inner_train
        )

        Xv_inner_val_imp = val_imputer_inner.transform(
            Xv_inner_val
        )


        # ----------------------------------------------------
        # Train the two independent classifiers
        # ----------------------------------------------------

        geo_model_inner = clone(base_xgb_model)
        val_model_inner = clone(base_xgb_model)

        geo_model_inner.fit(
            Xg_inner_train_imp,
            y_inner_train
        )

        val_model_inner.fit(
            Xv_inner_train_imp,
            y_inner_train
        )


        # ----------------------------------------------------
        # Inner validation probabilities
        # ----------------------------------------------------

        inner_geo_prob[inner_val_idx] = (
            geo_model_inner.predict_proba(
                Xg_inner_val_imp
            )[:, 1]
        )

        inner_val_prob[inner_val_idx] = (
            val_model_inner.predict_proba(
                Xv_inner_val_imp
            )[:, 1]
        )


    # ========================================================
    # 5. Optimize alpha + threshold using INNER OOF only
    # ========================================================

    best_score = -np.inf
    best_alpha = None
    best_threshold = None


    for alpha in alpha_grid:

        fused_inner_prob = (
            alpha * inner_geo_prob
            +
            (1 - alpha) * inner_val_prob
        )

        for threshold in threshold_grid:

            fused_inner_pred = (
                fused_inner_prob >= threshold
            ).astype(int)

            score = balanced_accuracy_score(
                y_train,
                fused_inner_pred
            )

            if score > best_score:

                best_score = score
                best_alpha = alpha
                best_threshold = threshold


    selected_alphas.append(best_alpha)
    selected_thresholds.append(best_threshold)


    # ========================================================
    # 6. Refit both stream models using ALL outer training data
    # ========================================================

    geo_imputer = SimpleImputer(strategy="mean")
    val_imputer = SimpleImputer(strategy="mean")

    Xg_train_imp = geo_imputer.fit_transform(Xg_train)
    Xg_test_imp = geo_imputer.transform(Xg_test)

    Xv_train_imp = val_imputer.fit_transform(Xv_train)
    Xv_test_imp = val_imputer.transform(Xv_test)


    geo_model = clone(base_xgb_model)
    val_model = clone(base_xgb_model)

    geo_model.fit(
        Xg_train_imp,
        y_train
    )

    val_model.fit(
        Xv_train_imp,
        y_train
    )


    # ========================================================
    # 7. Predict held-out participant
    # ========================================================

    geo_prob = geo_model.predict_proba(
        Xg_test_imp
    )[:, 1]

    val_prob = val_model.predict_proba(
        Xv_test_imp
    )[:, 1]


    fused_prob = (
        best_alpha * geo_prob
        +
        (1 - best_alpha) * val_prob
    )


    fused_pred = (
        fused_prob >= best_threshold
    ).astype(int)


    # ========================================================
    # Store GLOBAL OOF predictions
    # ========================================================

    all_true.extend(y_test)
    all_pred.extend(fused_pred)
    all_prob.extend(fused_prob)
    all_participants.extend(groups_test)


    if fold % 10 == 0 or fold == n_folds:

        print(
            f"Completed {fold}/{n_folds} | "
            f"alpha={best_alpha:.2f} | "
            f"threshold={best_threshold:.2f} | "
            f"inner BA={best_score:.4f}"
        )


# ============================================================
# 8. GLOBAL OOF evaluation
# ============================================================

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_prob = np.array(all_prob)


accuracy = accuracy_score(
    all_true,
    all_pred
)

balanced_acc = balanced_accuracy_score(
    all_true,
    all_pred
)

precision = precision_score(
    all_true,
    all_pred,
    zero_division=0
)

recall = recall_score(
    all_true,
    all_pred,
    zero_division=0
)

f1 = f1_score(
    all_true,
    all_pred,
    zero_division=0
)


results = pd.DataFrame({
    "Configuration": [
        "Optimized Weighted Late Fusion"
    ],
    "Accuracy": [
        f"{accuracy * 100:.2f}%"
    ],
    "Balanced Accuracy": [
        f"{balanced_acc * 100:.2f}%"
    ],
    "Precision": [
        f"{precision * 100:.2f}%"
    ],
    "Recall": [
        f"{recall * 100:.2f}%"
    ],
    "F1": [
        f"{f1 * 100:.2f}%"
    ],
    "N": [
        len(all_true)
    ]
})

display(results)


# ============================================================
# 9. Examine selected fusion parameters
# ============================================================

print("\nSelected alpha statistics")
print("-------------------------")
print(
    "Mean geometric weight:",
    np.mean(selected_alphas)
)

print(
    "Median geometric weight:",
    np.median(selected_alphas)
)

print(
    "Min geometric weight:",
    np.min(selected_alphas)
)

print(
    "Max geometric weight:",
    np.max(selected_alphas)
)


print("\nSelected threshold statistics")
print("-----------------------------")
print(
    "Mean threshold:",
    np.mean(selected_thresholds)
)

print(
    "Median threshold:",
    np.median(selected_thresholds)
)


# ------------------------------------------------------------
# optional detailed fold table
# ------------------------------------------------------------

parameter_df = pd.DataFrame({
    "Fold": np.arange(
        1,
        len(selected_alphas) + 1
    ),
    "Geometric Weight": selected_alphas,
    "Valence Weight": [
        1 - a for a in selected_alphas
    ],
    "Threshold": selected_thresholds
})

display(parameter_df)

LOSO folds: 56
Completed 10/56 | alpha=0.75 | threshold=0.70 | inner BA=0.9408
Completed 20/56 | alpha=0.90 | threshold=0.70 | inner BA=0.9391
Completed 30/56 | alpha=0.80 | threshold=0.70 | inner BA=0.9417
Completed 40/56 | alpha=0.80 | threshold=0.70 | inner BA=0.9411
Completed 50/56 | alpha=0.75 | threshold=0.70 | inner BA=0.9436
Completed 56/56 | alpha=0.80 | threshold=0.70 | inner BA=0.9477


,Configuration,Accuracy,Balanced Accuracy,Precision,Recall,F1,N
0,Optimized Weighted Late Fusion,93.98%,94.29%,97.51%,93.35%,95.38%,29736



Selected alpha statistics
-------------------------
Mean geometric weight: 0.7866071428571428
Median geometric weight: 0.75
Min geometric weight: 0.7000000000000001
Max geometric weight: 1.0

Selected threshold statistics
-----------------------------
Mean threshold: 0.6960714285714289
Median threshold: 0.7000000000000004


,Fold,Geometric Weight,Valence Weight,Threshold
0,1,1.00,0.00,0.70
1,2,0.75,0.25,0.70
2,3,0.85,0.15,0.70
3,4,0.85,0.15,0.70
4,5,0.75,0.25,0.70
5,6,0.70,0.30,0.68
6,7,0.80,0.20,0.70
7,8,0.80,0.20,0.70
8,9,0.85,0.15,0.70
9,10,0.75,0.25,0.70


In [4]:
# ============================================================
# IMPROVED LATE FUSION
# Optimized fusion weight alpha + optimized decision threshold
# Outer evaluation: LOSO
# Inner optimization: GroupKFold using TRAIN participants only
# Final metric: GLOBAL OOF Balanced Accuracy
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Canonical XGBoost model
# ------------------------------------------------------------

base_xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)


# ------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values


# ------------------------------------------------------------
# 3. Search space
# ------------------------------------------------------------

# alpha = geometric contribution
# 1-alpha = valence contribution

alpha_grid = np.arange(0.60, 1.001, 0.01)

threshold_grid = np.arange(0.50, 0.901, 0.01)


# ------------------------------------------------------------
# 4. Outer LOSO
# ------------------------------------------------------------

logo = LeaveOneGroupOut()

all_true = []
all_pred = []
all_prob = []
all_participants = []

selected_alphas = []
selected_thresholds = []

n_folds = logo.get_n_splits(X_geo, y, groups)

print("LOSO folds:", n_folds)


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_geo, y, groups), start=1
):

    # ========================================================
    # OUTER TRAIN / TEST
    # ========================================================

    Xg_train = X_geo.iloc[train_idx]
    Xg_test = X_geo.iloc[test_idx]

    Xv_train = X_val.iloc[train_idx]
    Xv_test = X_val.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]
    groups_test = groups[test_idx]


    # ========================================================
    # INNER GROUPED CV
    #
    # We generate OOF probabilities ONLY from outer-training
    # participants.
    # ========================================================

    unique_train_groups = np.unique(groups_train)

    n_inner_splits = min(5, len(unique_train_groups))

    inner_cv = GroupKFold(n_splits=n_inner_splits)

    inner_geo_prob = np.zeros(len(train_idx))
    inner_val_prob = np.zeros(len(train_idx))


    for inner_train_idx, inner_val_idx in inner_cv.split(
        Xg_train,
        y_train,
        groups_train
    ):

        # ----------------------------------------------------
        # Inner data
        # ----------------------------------------------------

        Xg_inner_train = Xg_train.iloc[inner_train_idx]
        Xg_inner_val = Xg_train.iloc[inner_val_idx]

        Xv_inner_train = Xv_train.iloc[inner_train_idx]
        Xv_inner_val = Xv_train.iloc[inner_val_idx]

        y_inner_train = y_train[inner_train_idx]


        # ----------------------------------------------------
        # Imputation fitted ONLY on inner-training subjects
        # ----------------------------------------------------

        geo_imputer_inner = SimpleImputer(strategy="mean")
        val_imputer_inner = SimpleImputer(strategy="mean")

        Xg_inner_train_imp = geo_imputer_inner.fit_transform(
            Xg_inner_train
        )

        Xg_inner_val_imp = geo_imputer_inner.transform(
            Xg_inner_val
        )

        Xv_inner_train_imp = val_imputer_inner.fit_transform(
            Xv_inner_train
        )

        Xv_inner_val_imp = val_imputer_inner.transform(
            Xv_inner_val
        )


        # ----------------------------------------------------
        # Train the two independent classifiers
        # ----------------------------------------------------

        geo_model_inner = clone(base_xgb_model)
        val_model_inner = clone(base_xgb_model)

        geo_model_inner.fit(
            Xg_inner_train_imp,
            y_inner_train
        )

        val_model_inner.fit(
            Xv_inner_train_imp,
            y_inner_train
        )


        # ----------------------------------------------------
        # Inner validation probabilities
        # ----------------------------------------------------

        inner_geo_prob[inner_val_idx] = (
            geo_model_inner.predict_proba(
                Xg_inner_val_imp
            )[:, 1]
        )

        inner_val_prob[inner_val_idx] = (
            val_model_inner.predict_proba(
                Xv_inner_val_imp
            )[:, 1]
        )


    # ========================================================
    # 5. Optimize alpha + threshold using INNER OOF only
    # ========================================================

    best_score = -np.inf
    best_alpha = None
    best_threshold = None


    for alpha in alpha_grid:

        fused_inner_prob = (
            alpha * inner_geo_prob
            +
            (1 - alpha) * inner_val_prob
        )

        for threshold in threshold_grid:

            fused_inner_pred = (
                fused_inner_prob >= threshold
            ).astype(int)

            score = balanced_accuracy_score(
                y_train,
                fused_inner_pred
            )

            if score > best_score:

                best_score = score
                best_alpha = alpha
                best_threshold = threshold


    selected_alphas.append(best_alpha)
    selected_thresholds.append(best_threshold)


    # ========================================================
    # 6. Refit both stream models using ALL outer training data
    # ========================================================

    geo_imputer = SimpleImputer(strategy="mean")
    val_imputer = SimpleImputer(strategy="mean")

    Xg_train_imp = geo_imputer.fit_transform(Xg_train)
    Xg_test_imp = geo_imputer.transform(Xg_test)

    Xv_train_imp = val_imputer.fit_transform(Xv_train)
    Xv_test_imp = val_imputer.transform(Xv_test)


    geo_model = clone(base_xgb_model)
    val_model = clone(base_xgb_model)

    geo_model.fit(
        Xg_train_imp,
        y_train
    )

    val_model.fit(
        Xv_train_imp,
        y_train
    )


    # ========================================================
    # 7. Predict held-out participant
    # ========================================================

    geo_prob = geo_model.predict_proba(
        Xg_test_imp
    )[:, 1]

    val_prob = val_model.predict_proba(
        Xv_test_imp
    )[:, 1]


    fused_prob = (
        best_alpha * geo_prob
        +
        (1 - best_alpha) * val_prob
    )


    fused_pred = (
        fused_prob >= best_threshold
    ).astype(int)


    # ========================================================
    # Store GLOBAL OOF predictions
    # ========================================================

    all_true.extend(y_test)
    all_pred.extend(fused_pred)
    all_prob.extend(fused_prob)
    all_participants.extend(groups_test)


    if fold % 10 == 0 or fold == n_folds:

        print(
            f"Completed {fold}/{n_folds} | "
            f"alpha={best_alpha:.2f} | "
            f"threshold={best_threshold:.2f} | "
            f"inner BA={best_score:.4f}"
        )


# ============================================================
# 8. GLOBAL OOF evaluation
# ============================================================

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_prob = np.array(all_prob)


accuracy = accuracy_score(
    all_true,
    all_pred
)

balanced_acc = balanced_accuracy_score(
    all_true,
    all_pred
)

precision = precision_score(
    all_true,
    all_pred,
    zero_division=0
)

recall = recall_score(
    all_true,
    all_pred,
    zero_division=0
)

f1 = f1_score(
    all_true,
    all_pred,
    zero_division=0
)


results = pd.DataFrame({
    "Configuration": [
        "Optimized Weighted Late Fusion"
    ],
    "Accuracy": [
        f"{accuracy * 100:.2f}%"
    ],
    "Balanced Accuracy": [
        f"{balanced_acc * 100:.2f}%"
    ],
    "Precision": [
        f"{precision * 100:.2f}%"
    ],
    "Recall": [
        f"{recall * 100:.2f}%"
    ],
    "F1": [
        f"{f1 * 100:.2f}%"
    ],
    "N": [
        len(all_true)
    ]
})

display(results)


# ============================================================
# 9. Examine selected fusion parameters
# ============================================================

print("\nSelected alpha statistics")
print("-------------------------")
print(
    "Mean geometric weight:",
    np.mean(selected_alphas)
)

print(
    "Median geometric weight:",
    np.median(selected_alphas)
)

print(
    "Min geometric weight:",
    np.min(selected_alphas)
)

print(
    "Max geometric weight:",
    np.max(selected_alphas)
)


print("\nSelected threshold statistics")
print("-----------------------------")
print(
    "Mean threshold:",
    np.mean(selected_thresholds)
)

print(
    "Median threshold:",
    np.median(selected_thresholds)
)


# ------------------------------------------------------------
# optional detailed fold table
# ------------------------------------------------------------

parameter_df = pd.DataFrame({
    "Fold": np.arange(
        1,
        len(selected_alphas) + 1
    ),
    "Geometric Weight": selected_alphas,
    "Valence Weight": [
        1 - a for a in selected_alphas
    ],
    "Threshold": selected_thresholds
})

display(parameter_df)

LOSO folds: 56
Completed 10/56 | alpha=0.80 | threshold=0.76 | inner BA=0.9414
Completed 20/56 | alpha=0.78 | threshold=0.75 | inner BA=0.9408
Completed 30/56 | alpha=0.85 | threshold=0.69 | inner BA=0.9418
Completed 40/56 | alpha=0.89 | threshold=0.77 | inner BA=0.9421
Completed 50/56 | alpha=0.90 | threshold=0.79 | inner BA=0.9463
Completed 56/56 | alpha=0.91 | threshold=0.77 | inner BA=0.9485


,Configuration,Accuracy,Balanced Accuracy,Precision,Recall,F1,N
0,Optimized Weighted Late Fusion,93.66%,94.27%,97.94%,92.43%,95.11%,29736



Selected alpha statistics
-------------------------
Mean geometric weight: 0.8728571428571431
Median geometric weight: 0.8650000000000002
Min geometric weight: 0.68
Max geometric weight: 1.0000000000000004

Selected threshold statistics
-----------------------------
Mean threshold: 0.7682142857142861
Median threshold: 0.7700000000000002


,Fold,Geometric Weight,Valence Weight,Threshold
0,1,0.83,1.700000e-01,0.76
1,2,0.83,1.700000e-01,0.74
2,3,0.90,1.000000e-01,0.79
3,4,0.93,7.000000e-02,0.74
4,5,0.82,1.800000e-01,0.72
5,6,0.68,3.200000e-01,0.68
6,7,0.81,1.900000e-01,0.69
7,8,0.88,1.200000e-01,0.74
8,9,0.86,1.400000e-01,0.69
9,10,0.80,2.000000e-01,0.76


In [5]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


# ============================================================
# XGBOOST ONLY — OPTIMIZED LATE FUSION VERIFICATION
# ============================================================

base_xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values

alpha_grid = np.arange(0.60, 1.001, 0.01)
threshold_grid = np.arange(0.50, 0.901, 0.01)

logo = LeaveOneGroupOut()

all_true = []
all_pred = []

selected_alphas = []
selected_thresholds = []

n_folds = logo.get_n_splits(X_geo, y, groups)

print("LOSO folds:", n_folds)

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_geo, y, groups),
    start=1
):

    Xg_train = X_geo.iloc[train_idx]
    Xg_test = X_geo.iloc[test_idx]

    Xv_train = X_val.iloc[train_idx]
    Xv_test = X_val.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]

    # --------------------------------------------------------
    # INNER GROUPED CV
    # --------------------------------------------------------

    inner_cv = GroupKFold(n_splits=5)

    inner_geo_prob = np.zeros(len(train_idx))
    inner_val_prob = np.zeros(len(train_idx))

    for inner_train_idx, inner_val_idx in inner_cv.split(
        Xg_train,
        y_train,
        groups_train
    ):

        geo_imputer_inner = SimpleImputer(strategy="mean")
        val_imputer_inner = SimpleImputer(strategy="mean")

        Xg_inner_train = geo_imputer_inner.fit_transform(
            Xg_train.iloc[inner_train_idx]
        )

        Xg_inner_val = geo_imputer_inner.transform(
            Xg_train.iloc[inner_val_idx]
        )

        Xv_inner_train = val_imputer_inner.fit_transform(
            Xv_train.iloc[inner_train_idx]
        )

        Xv_inner_val = val_imputer_inner.transform(
            Xv_train.iloc[inner_val_idx]
        )

        y_inner_train = y_train[inner_train_idx]

        geo_model_inner = clone(base_xgb_model)
        val_model_inner = clone(base_xgb_model)

        geo_model_inner.fit(
            Xg_inner_train,
            y_inner_train
        )

        val_model_inner.fit(
            Xv_inner_train,
            y_inner_train
        )

        inner_geo_prob[inner_val_idx] = (
            geo_model_inner.predict_proba(
                Xg_inner_val
            )[:, 1]
        )

        inner_val_prob[inner_val_idx] = (
            val_model_inner.predict_proba(
                Xv_inner_val
            )[:, 1]
        )


    # --------------------------------------------------------
    # OPTIMIZE ALPHA + THRESHOLD
    # --------------------------------------------------------

    best_score = -np.inf
    best_alpha = None
    best_threshold = None

    for alpha in alpha_grid:

        fused_inner_prob = (
            alpha * inner_geo_prob
            +
            (1 - alpha) * inner_val_prob
        )

        for threshold in threshold_grid:

            fused_inner_pred = (
                fused_inner_prob >= threshold
            ).astype(int)

            score = balanced_accuracy_score(
                y_train,
                fused_inner_pred
            )

            if score > best_score:
                best_score = score
                best_alpha = alpha
                best_threshold = threshold

    selected_alphas.append(best_alpha)
    selected_thresholds.append(best_threshold)


    # --------------------------------------------------------
    # REFIT ON ALL OUTER TRAINING DATA
    # --------------------------------------------------------

    geo_imputer = SimpleImputer(strategy="mean")
    val_imputer = SimpleImputer(strategy="mean")

    Xg_train_imp = geo_imputer.fit_transform(Xg_train)
    Xg_test_imp = geo_imputer.transform(Xg_test)

    Xv_train_imp = val_imputer.fit_transform(Xv_train)
    Xv_test_imp = val_imputer.transform(Xv_test)

    geo_model = clone(base_xgb_model)
    val_model = clone(base_xgb_model)

    geo_model.fit(
        Xg_train_imp,
        y_train
    )

    val_model.fit(
        Xv_train_imp,
        y_train
    )

    geo_prob = geo_model.predict_proba(
        Xg_test_imp
    )[:, 1]

    val_prob = val_model.predict_proba(
        Xv_test_imp
    )[:, 1]

    fused_prob = (
        best_alpha * geo_prob
        +
        (1 - best_alpha) * val_prob
    )

    fused_pred = (
        fused_prob >= best_threshold
    ).astype(int)

    all_true.extend(y_test)
    all_pred.extend(fused_pred)

    if fold % 10 == 0 or fold == n_folds:
        print(
            f"Completed {fold}/{n_folds} | "
            f"alpha={best_alpha:.2f} | "
            f"threshold={best_threshold:.2f}"
        )


# ============================================================
# FINAL GLOBAL OOF RESULTS
# ============================================================

all_true = np.array(all_true)
all_pred = np.array(all_pred)

accuracy = accuracy_score(all_true, all_pred)
ba = balanced_accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred)
recall = recall_score(all_true, all_pred)
f1 = f1_score(all_true, all_pred)

print("\nFINAL XGBOOST OPTIMIZED LATE FUSION")
print("-------------------------------------")
print(f"Accuracy:          {accuracy * 100:.2f}%")
print(f"Balanced Accuracy: {ba * 100:.2f}%")
print(f"Precision:         {precision * 100:.2f}%")
print(f"Recall:            {recall * 100:.2f}%")
print(f"F1-score:          {f1 * 100:.2f}%")

print("\nMean geometric weight:",
      np.mean(selected_alphas))

print("Mean threshold:",
      np.mean(selected_thresholds))

LOSO folds: 56
Completed 10/56 | alpha=0.80 | threshold=0.76
Completed 20/56 | alpha=0.78 | threshold=0.75
Completed 30/56 | alpha=0.85 | threshold=0.69
Completed 40/56 | alpha=0.89 | threshold=0.77
Completed 50/56 | alpha=0.90 | threshold=0.79
Completed 56/56 | alpha=0.91 | threshold=0.77

FINAL XGBOOST OPTIMIZED LATE FUSION
-------------------------------------
Accuracy:          93.66%
Balanced Accuracy: 94.27%
Precision:         97.94%
Recall:            92.43%
F1-score:          95.11%

Mean geometric weight: 0.8728571428571431
Mean threshold: 0.7682142857142861


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    r"C:\Users\hp\StressDetection\master_features_restored.csv"
)

geo_features = [
    "EAR_mean", "EAR_std", "EAR_min", "EAR_max", "EAR_range",
    "blink_mean", "blink_std", "blink_min", "blink_max", "blink_range",
    "gaze_mean", "gaze_std", "gaze_min", "gaze_max", "gaze_range",
    "brow_mean", "brow_std", "brow_min", "brow_max", "brow_range",
    "lip_mean", "lip_std", "lip_min", "lip_max", "lip_range"
]

valence_features = [
    "valence_mean",
    "valence_std",
    "valence_min",
    "valence_max"
]

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values


# ============================================================
# FUSION SEARCH SPACE
# ============================================================

alpha_grid = np.arange(0.60, 1.001, 0.01)
threshold_grid = np.arange(0.50, 0.901, 0.01)


# ============================================================
# CLASSIFIERS
# ============================================================

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=42
    )
}


# ============================================================
# NESTED LOSO FUNCTION
# ============================================================

def run_nested_loso(model_name, base_model):

    logo = LeaveOneGroupOut()

    all_true = []
    all_pred = []

    selected_alphas = []
    selected_thresholds = []

    outer_splits = list(
        logo.split(X_geo, y, groups)
    )

    print(f"\n===== {model_name} =====")
    print(f"LOSO folds: {len(outer_splits)}")

    # ========================================================
    # OUTER LOSO LOOP
    # ========================================================

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_splits,
        start=1
    ):

        # ----------------------------------------------------
        # Outer training / testing data
        # ----------------------------------------------------

        Xg_train = X_geo.iloc[train_idx]
        Xg_test = X_geo.iloc[test_idx]

        Xv_train = X_val.iloc[train_idx]
        Xv_test = X_val.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        groups_train = groups[train_idx]

        # ====================================================
        # INNER PARTICIPANT-GROUPED CROSS-VALIDATION
        # ====================================================

        inner_cv = GroupKFold(n_splits=5)

        inner_p_geo = np.zeros(len(train_idx))
        inner_p_val = np.zeros(len(train_idx))

        for inner_train_idx, inner_val_idx in inner_cv.split(
            Xg_train,
            y_train,
            groups_train
        ):

            # -----------------------------------------------
            # Separate model for geometric stream
            # -----------------------------------------------

            geo_pipe = Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="mean")
                ),
                (
                    "model",
                    clone(base_model)
                )
            ])

            # -----------------------------------------------
            # Separate model for valence stream
            # -----------------------------------------------

            val_pipe = Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="mean")
                ),
                (
                    "model",
                    clone(base_model)
                )
            ])

            # -----------------------------------------------
            # Fit geometric model
            # -----------------------------------------------

            geo_pipe.fit(
                Xg_train.iloc[inner_train_idx],
                y_train[inner_train_idx]
            )

            # -----------------------------------------------
            # Fit valence model
            # -----------------------------------------------

            val_pipe.fit(
                Xv_train.iloc[inner_train_idx],
                y_train[inner_train_idx]
            )

            # -----------------------------------------------
            # Inner validation probabilities
            # -----------------------------------------------

            inner_p_geo[inner_val_idx] = (
                geo_pipe.predict_proba(
                    Xg_train.iloc[inner_val_idx]
                )[:, 1]
            )

            inner_p_val[inner_val_idx] = (
                val_pipe.predict_proba(
                    Xv_train.iloc[inner_val_idx]
                )[:, 1]
            )

        # ====================================================
        # OPTIMIZE FUSION WEIGHT + DECISION THRESHOLD
        # ====================================================

        best_ba = -1
        best_alpha = None
        best_threshold = None

        for alpha in alpha_grid:

            fused_inner = (
                alpha * inner_p_geo
                + (1 - alpha) * inner_p_val
            )

            for threshold in threshold_grid:

                inner_pred = (
                    fused_inner >= threshold
                ).astype(int)

                ba = balanced_accuracy_score(
                    y_train,
                    inner_pred
                )

                if ba > best_ba:
                    best_ba = ba
                    best_alpha = alpha
                    best_threshold = threshold

        selected_alphas.append(best_alpha)
        selected_thresholds.append(best_threshold)

        # ====================================================
        # REFIT MODELS ON FULL OUTER TRAINING SET
        # ====================================================

        final_geo_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "model",
                clone(base_model)
            )
        ])

        final_val_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "model",
                clone(base_model)
            )
        ])

        final_geo_pipe.fit(
            Xg_train,
            y_train
        )

        final_val_pipe.fit(
            Xv_train,
            y_train
        )

        # ====================================================
        # TEST ON HELD-OUT PARTICIPANT
        # ====================================================

        p_geo_test = final_geo_pipe.predict_proba(
            Xg_test
        )[:, 1]

        p_val_test = final_val_pipe.predict_proba(
            Xv_test
        )[:, 1]

        fused_test = (
            best_alpha * p_geo_test
            + (1 - best_alpha) * p_val_test
        )

        y_pred = (
            fused_test >= best_threshold
        ).astype(int)

        all_true.extend(y_test)
        all_pred.extend(y_pred)

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            fold_idx % 10 == 0
            or fold_idx == len(outer_splits)
        ):
            print(
                f"Completed {fold_idx}/{len(outer_splits)} | "
                f"alpha={best_alpha:.2f} | "
                f"threshold={best_threshold:.2f}"
            )

    # ========================================================
    # GLOBAL OUT-OF-FOLD METRICS
    # ========================================================

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        all_true,
        all_pred
    )

    precision = precision_score(
        all_true,
        all_pred
    )

    recall = recall_score(
        all_true,
        all_pred
    )

    f1 = f1_score(
        all_true,
        all_pred
    )

    mean_geo_weight = np.mean(
        selected_alphas
    )

    mean_val_weight = (
        1 - mean_geo_weight
    )

    mean_threshold = np.mean(
        selected_thresholds
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"\nFINAL {model_name.upper()} "
        f"OPTIMIZED LATE FUSION"
    )

    print(
        f"Accuracy:          "
        f"{accuracy * 100:.2f}%"
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_accuracy * 100:.2f}%"
    )

    print(
        f"Precision:         "
        f"{precision * 100:.2f}%"
    )

    print(
        f"Recall:            "
        f"{recall * 100:.2f}%"
    )

    print(
        f"F1-score:          "
        f"{f1 * 100:.2f}%"
    )

    print(
        f"\nMean geometric weight: "
        f"{mean_geo_weight:.4f}"
    )

    print(
        f"Mean valence weight:   "
        f"{mean_val_weight:.4f}"
    )

    print(
        f"Mean threshold:        "
        f"{mean_threshold:.4f}"
    )

    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {
        "classifier": model_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_geo_weight": mean_geo_weight,
        "mean_val_weight": mean_val_weight,
        "mean_threshold": mean_threshold
    }


# ============================================================
# RUN RANDOM FOREST
# ============================================================

rf_results = run_nested_loso(
    "Random Forest",
    models["Random Forest"]
)


# ============================================================
# RUN LINEAR SVM
# ============================================================

svm_results = run_nested_loso(
    "Linear SVM",
    models["Linear SVM"]
)


# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame([
    rf_results,
    svm_results
])

display(comparison)


===== Random Forest =====
LOSO folds: 56
Completed 10/56 | alpha=0.83 | threshold=0.71
Completed 20/56 | alpha=0.78 | threshold=0.69
Completed 30/56 | alpha=0.88 | threshold=0.70
Completed 40/56 | alpha=0.84 | threshold=0.72
Completed 50/56 | alpha=0.95 | threshold=0.74
Completed 56/56 | alpha=0.84 | threshold=0.72

FINAL RANDOM FOREST OPTIMIZED LATE FUSION
Accuracy:          92.40%
Balanced Accuracy: 93.12%
Precision:         97.48%
Recall:            90.96%
F1-score:          94.11%

Mean geometric weight: 0.8302
Mean valence weight:   0.1698
Mean threshold:        0.7157

===== Linear SVM =====
LOSO folds: 56


In [1]:
# ============================================================
# RUN LINEAR SVM ONLY
# ============================================================

svm_results = run_nested_loso(
    "Linear SVM",
    models["Linear SVM"]
)

print("\n==============================")
print("LINEAR SVM FINAL RESULTS")
print("==============================")

display(pd.DataFrame([svm_results]))

NameError: name 'run_nested_loso' is not defined

In [ ]:
# ============================================================
# LINEAR SVM - NESTED LOSO WITH OPTIMIZED LATE FUSION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.svm import SVC


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    r"C:\Users\hp\StressDetection\master_features_restored.csv"
)


# ============================================================
# FEATURES
# ============================================================

geo_features = [
    "EAR_mean", "EAR_std", "EAR_min", "EAR_max", "EAR_range",
    "blink_mean", "blink_std", "blink_min", "blink_max", "blink_range",
    "gaze_mean", "gaze_std", "gaze_min", "gaze_max", "gaze_range",
    "brow_mean", "brow_std", "brow_min", "brow_max", "brow_range",
    "lip_mean", "lip_std", "lip_min", "lip_max", "lip_range"
]

valence_features = [
    "valence_mean",
    "valence_std",
    "valence_min",
    "valence_max"
]

X_geo = df[geo_features].copy()
X_val = df[valence_features].copy()

y = df["label"].values
groups = df["participant"].values


# ============================================================
# FUSION SEARCH SPACE
# ============================================================

alpha_grid = np.arange(0.60, 1.001, 0.01)
threshold_grid = np.arange(0.50, 0.901, 0.01)


# ============================================================
# LINEAR SVM
# ============================================================

base_model = SVC(
    kernel="linear",
    probability=True,
    random_state=42
)


# ============================================================
# NESTED LOSO FUNCTION
# ============================================================

def run_nested_loso_svm():

    logo = LeaveOneGroupOut()

    all_true = []
    all_pred = []

    selected_alphas = []
    selected_thresholds = []

    outer_splits = list(
        logo.split(X_geo, y, groups)
    )

    print("\n===== LINEAR SVM =====")
    print(f"LOSO folds: {len(outer_splits)}")


    # ========================================================
    # OUTER LOSO LOOP
    # ========================================================

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_splits,
        start=1
    ):

        # ----------------------------------------------------
        # Outer training / testing
        # ----------------------------------------------------

        Xg_train = X_geo.iloc[train_idx]
        Xg_test = X_geo.iloc[test_idx]

        Xv_train = X_val.iloc[train_idx]
        Xv_test = X_val.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        groups_train = groups[train_idx]


        # ====================================================
        # INNER GROUPED CROSS-VALIDATION
        # ====================================================

        inner_cv = GroupKFold(n_splits=5)

        inner_p_geo = np.zeros(len(train_idx))
        inner_p_val = np.zeros(len(train_idx))


        for inner_train_idx, inner_val_idx in inner_cv.split(
            Xg_train,
            y_train,
            groups_train
        ):

            # ------------------------------------------------
            # GEOMETRIC STREAM PIPELINE
            # ------------------------------------------------

            geo_pipe = Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="mean")
                ),
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "model",
                    clone(base_model)
                )
            ])


            # ------------------------------------------------
            # VALENCE STREAM PIPELINE
            # ------------------------------------------------

            val_pipe = Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="mean")
                ),
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "model",
                    clone(base_model)
                )
            ])


            # ------------------------------------------------
            # FIT GEOMETRIC STREAM
            # ------------------------------------------------

            geo_pipe.fit(
                Xg_train.iloc[inner_train_idx],
                y_train[inner_train_idx]
            )


            # ------------------------------------------------
            # FIT VALENCE STREAM
            # ------------------------------------------------

            val_pipe.fit(
                Xv_train.iloc[inner_train_idx],
                y_train[inner_train_idx]
            )


            # ------------------------------------------------
            # INNER VALIDATION PROBABILITIES
            # ------------------------------------------------

            inner_p_geo[inner_val_idx] = (
                geo_pipe.predict_proba(
                    Xg_train.iloc[inner_val_idx]
                )[:, 1]
            )

            inner_p_val[inner_val_idx] = (
                val_pipe.predict_proba(
                    Xv_train.iloc[inner_val_idx]
                )[:, 1]
            )


        # ====================================================
        # OPTIMIZE FUSION WEIGHT + THRESHOLD
        # ====================================================

        best_ba = -1
        best_alpha = None
        best_threshold = None

        for alpha in alpha_grid:

            fused_inner = (
                alpha * inner_p_geo
                + (1 - alpha) * inner_p_val
            )

            for threshold in threshold_grid:

                inner_pred = (
                    fused_inner >= threshold
                ).astype(int)

                ba = balanced_accuracy_score(
                    y_train,
                    inner_pred
                )

                if ba > best_ba:
                    best_ba = ba
                    best_alpha = alpha
                    best_threshold = threshold


        selected_alphas.append(best_alpha)
        selected_thresholds.append(best_threshold)


        # ====================================================
        # REFIT ON FULL OUTER TRAINING SET
        # ====================================================

        final_geo_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                clone(base_model)
            )
        ])

        final_val_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                clone(base_model)
            )
        ])


        final_geo_pipe.fit(
            Xg_train,
            y_train
        )

        final_val_pipe.fit(
            Xv_train,
            y_train
        )


        # ====================================================
        # TEST HELD-OUT PARTICIPANT
        # ====================================================

        p_geo_test = final_geo_pipe.predict_proba(
            Xg_test
        )[:, 1]

        p_val_test = final_val_pipe.predict_proba(
            Xv_test
        )[:, 1]


        fused_test = (
            best_alpha * p_geo_test
            + (1 - best_alpha) * p_val_test
        )


        y_pred = (
            fused_test >= best_threshold
        ).astype(int)


        all_true.extend(y_test)
        all_pred.extend(y_pred)


        # ====================================================
        # PROGRESS
        # ====================================================

        if (
            fold_idx % 10 == 0
            or fold_idx == len(outer_splits)
        ):

            print(
                f"Completed {fold_idx}/{len(outer_splits)} | "
                f"alpha={best_alpha:.2f} | "
                f"threshold={best_threshold:.2f}"
            )


    # ========================================================
    # GLOBAL METRICS
    # ========================================================

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)


    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        all_true,
        all_pred
    )

    precision = precision_score(
        all_true,
        all_pred,
        zero_division=0
    )

    recall = recall_score(
        all_true,
        all_pred,
        zero_division=0
    )

    f1 = f1_score(
        all_true,
        all_pred,
        zero_division=0
    )


    mean_geo_weight = np.mean(
        selected_alphas
    )

    mean_val_weight = (
        1 - mean_geo_weight
    )

    mean_threshold = np.mean(
        selected_thresholds
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n======================================")
    print("FINAL LINEAR SVM OPTIMIZED LATE FUSION")
    print("======================================")

    print(
        f"Accuracy:          "
        f"{accuracy * 100:.2f}%"
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_accuracy * 100:.2f}%"
    )

    print(
        f"Precision:         "
        f"{precision * 100:.2f}%"
    )

    print(
        f"Recall:            "
        f"{recall * 100:.2f}%"
    )

    print(
        f"F1-score:          "
        f"{f1 * 100:.2f}%"
    )

    print(
        f"\nMean geometric weight: "
        f"{mean_geo_weight:.4f}"
    )

    print(
        f"Mean valence weight:   "
        f"{mean_val_weight:.4f}"
    )

    print(
        f"Mean threshold:        "
        f"{mean_threshold:.4f}"
    )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {
        "classifier": "Linear SVM",
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_geo_weight": mean_geo_weight,
        "mean_val_weight": mean_val_weight,
        "mean_threshold": mean_threshold
    }


# ============================================================
# RUN LINEAR SVM
# ============================================================

svm_results = run_nested_loso_svm()


# ============================================================
# RESULTS TABLE
# ============================================================

svm_results_df = pd.DataFrame([svm_results])

display(svm_results_df)


===== LINEAR SVM =====
LOSO folds: 56
